# Local Taxes
Taxes collected by cities and counties, by type. SCO classifies franchise fees under taxes; they're shown separately here. 'Property tax in lieu of VLF' replaced the vehicle license fee revenue the state cut in 2004.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
SCO_NOTE = "Source: California State Controller's Office, Cities / Counties / Special Districts Annual Reports"
BREAK = {2016.5: "SCO reporting format changes"}
lt = pd.read_csv(PROCESSED / "local_taxes.csv")
for govt in ["City", "County"]:
    d = add_views(lt[lt.govt == govt].groupby(["fy", "tax_type"], as_index=False).amount_k.sum())
    stacked_area_toggle(d, "tax_type", f"{govt} tax revenue by type", SCO_NOTE).show()

## State and local taxes together
State taxes (all funds) next to city and county taxes. Excludes school districts (see the school page) and special districts, so local taxes are understated.

In [2]:
st = pd.read_csv(PROCESSED / "state_revenue.csv")
st = st[st.group == "Tax"].groupby("fy", as_index=False).amount_k.sum().assign(level="State")
loc = lt.groupby(["fy", "govt"], as_index=False).amount_k.sum().rename(columns={"govt": "level"})
loc["level"] = loc["level"] + " governments"
d = add_views(pd.concat([st, loc[["fy", "level", "amount_k"]]]))
d = d[d.fy.between(2003, LAST_ACTUAL_FY)]
stacked_area_toggle(d, "level", "State vs city and county taxes", "State = all funds (LAO / DOF). Local = SCO.",
                    order=["State", "County governments", "City governments"]).show()

## Property tax by recipient
Property tax collected by each kind of local government, 2003 on. The school-district series is from the Census (see the school page); city and county are SCO. Special districts (fire, water, parks) and community colleges also get property tax but are not separable in the data used here, so the total is understated. 'Property tax in lieu of VLF' is excluded.

In [3]:
sch = pd.read_csv(PROCESSED / "school_finance.csv")
pt = pd.concat([
    lt[lt.tax_type == "Property tax"].assign(recipient=lambda x: x.govt + " governments")[["fy", "recipient", "amount_k"]],
    sch[(sch.kind == "revenue") & (sch.group == "Local property tax")].assign(recipient="School districts")[["fy", "recipient", "amount_k"]],
])
pt = add_views(pt[pt.fy >= 2003])
stacked_area_toggle(pt, "recipient", "Property tax by recipient: schools, counties and cities",
                    "City and county: SCO. School districts: Census. Excludes special districts and community colleges.",
                    order=["School districts", "County governments", "City governments"]).show()